# Reddit Data Processing (Clean & Simple)
Spark pipeline to read XMLs from ZIPs, clean text, sorted export to CSV.

In [12]:
import os
import glob
import zipfile
import xml.etree.ElementTree as ET
import re
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, trim, length, lit
from pyspark.sql.types import StructType, StructField, StringType

# --- 1. CONFIGURATION ---
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jre-1.8"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]
# Point to Python executable for workers
PY_PATH = r"C:\Users\Franco\Desktop\PROGET~1\.venv\Scripts\python.exe"
os.environ['PYSPARK_PYTHON'] = PY_PATH
os.environ['PYSPARK_DRIVER_PYTHON'] = PY_PATH

spark = SparkSession.builder.master("local[*]").appName("RedditClean").getOrCreate()

In [13]:
# --- 2. EXTRACT (Read ZIPs) ---
def extract_xml_from_zip(filepath):
    """Reads a ZIP file and extracts content from all XMLs inside."""
    results = []
    try:
        chunk = re.search(r'chunk(\d+)', filepath, re.IGNORECASE).group(1) if 'chunk' in filepath.lower() else "Unknown"
        with zipfile.ZipFile(filepath, "r") as z:
            for filename in [f for f in z.namelist() if f.endswith('.xml')]:
                with z.open(filename) as f:
                    root = ET.fromstring(f.read())
                    
                    # Parse ID
                    raw_id = root.find('ID').text.strip()
                    subj_id = raw_id.replace("subject", "")
                    
                    # Parse Writings
                    for w in root.findall('WRITING'):
                        results.append((
                            subj_id, 
                            chunk, 
                            (w.find('DATE').text or "").strip(),
                            (w.find('TITLE').text or "").strip(),
                            (w.find('INFO').text or "").strip(),
                            (w.find('TEXT').text or "").strip()
                        ))
    except Exception:
        pass # Skip unreadable files
    return results

# Run Extraction in Parallel
zip_files = glob.glob(os.path.join("data", "*.zip"))
rdd = spark.sparkContext.parallelize(zip_files).flatMap(extract_xml_from_zip)

df_raw = spark.createDataFrame(rdd, ["SubjectID", "Chunk", "Date", "Title_Raw", "Info", "Text_Raw"])
print(f"Loaded {df_raw.count()} raw records.")

Loaded 544447 raw records.


In [14]:
# --- 3. TRANSFORM (Clean Text) ---
# Define regex patterns once for readability
REGEX_NEWLINE = r"[\r\n]+"
REGEX_MD_LINK = r"\[(.*?)\]\(.*?\)"  # Matches [Label](URL) -> keeps Label $1
REGEX_URL     = r"http\S+"

# Clean using 'select' for a clear columns definition
df_clean = df_raw.select(
    col("SubjectID"),
    col("Chunk"),
    col("Date"),
    col("Info"),
    
    # CLEAN TITLE
    trim(regexp_replace(
        regexp_replace(
            regexp_replace("Title_Raw", REGEX_NEWLINE, " "), 
        REGEX_MD_LINK, "$1"), 
    REGEX_URL, "")).alias("Title"),
    
    # CLEAN TEXT
    trim(regexp_replace(
        regexp_replace(
            regexp_replace("Text_Raw", REGEX_NEWLINE, " "), 
        REGEX_MD_LINK, "$1"), 
    REGEX_URL, "")).alias("Text")
).filter(length(col("Text")) > 0)

df_clean.show(5, truncate=50)

+---------+-----+-------------------+-----------+--------------------------------------------------+--------------------------------------------------+
|SubjectID|Chunk|               Date|       Info|                                             Title|                                              Text|
+---------+-----+-------------------+-----------+--------------------------------------------------+--------------------------------------------------+
|     1010|    1|2016-01-31 02:50:20|reddit post|What did Bernie Sanders say when he found a dea...|"Oh god, a caucus!" ^^^cuz ^^^he ^^^has ^^^a ^^...|
|     1010|    1|2016-01-30 19:52:13|reddit post|                                                  |I didn't have any trouble with faux iosefka but...|
|     1010|    1|2016-01-30 18:40:43|reddit post|                                                  |Did you have trouble with Yurie and Faux Iosefk...|
|     1010|    1|2016-01-30 14:52:54|reddit post|                                       

In [15]:
# --- 4. ENRICH (Add Labels) ---
# Read Labels
df_labels = spark.read.option("delimiter", "\t").csv(os.path.join("data", "risk-golden-truth-test.txt")) \
    .select(regexp_replace("_c0", "subject", "").alias("SubjectID"), col("_c1").alias("Label"))

# Join
df_final = df_clean.join(df_labels, "SubjectID", "left")

In [16]:
# --- 5. EXPORT (Windows-Safe) ---
import csv

output_file = os.path.join("data", "reddit_clean_dataset.csv")
print(f"Saving to {output_file} (Streaming to avoid OOM & Winutils)...")

# Use toLocalIterator to iterate data into the driver row-by-row
# This bypasses the need for Hadoop/Winutils.exe on Windows
with open(output_file, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f, quoting=csv.QUOTE_MINIMAL)
    
    # Write Header
    writer.writerow(df_final.columns)
    
    # Write Rows
    # Sort by SubjectID (as Integer) and Date
    # casting to int ensures '2' comes before '10'
    sorted_df = df_final.orderBy(col("SubjectID").cast("int"), col("Date"))
    
    count = 0
    for row in sorted_df.toLocalIterator():
        writer.writerow(row)
        count += 1
        if count % 1000 == 0:
            print(f"Written {count} rows...", end="\r")
    
print(f"\nDone. Total: {count}")
spark.stop()

Saving to data\reddit_clean_dataset.csv (Streaming to avoid OOM & Winutils)...
Written 394000 rows...
Done. Total: 394915
